# 11 · 直接指挥每一个电机：位置、速度、力矩

**Part 2 用 SDK2 驱动机器人 · 第 7 节（G1，并与 Go2 对照）**

到目前为止，平衡都是机器人自己的运控在管。这一节把运控**关掉**，你来给每个电机发指令。
这是做强化学习部署、自研步态、柔顺控制的入口，也是最容易把机器人弄坏的地方。

一个问题贯穿全节：**我让关节去 1.0 rad，它为什么停在了 0.9？又为什么有时会抖？**

学完你能：
1. 填出一条合法的 `LowCmd`（含 `mode_machine` 和 CRC），说出每个字段的作用；
2. 用 `kp / kd / tau` 解释跟踪误差、振荡、稳态误差三种现象；
3. 用"限位 → 插值 → 限速"三道保险生成安全的目标；
4. 说出 G1 与 Go2 的 `LowCmd` 的四处不同。

> **低层控制时机器人没有平衡能力。** G1 必须吊在保护架上，Go2 必须趴着或吊着。本节默认只打印不下发。

In [ ]:
LIVE = False

import struct
import time
import numpy as np
import matplotlib.pyplot as plt

from common import connect, MotionGate
from common.bagio import load_npz
from common.robots import (G1_JOINTS, G1_JOINT_ID, G1_KP, G1_KD, G1_NUM_MOTOR, g1_limits,
                           GO2_JOINTS, GO2_POSE_STAND, GO2_LOWCMD_HEAD, GO2_LEVEL_LOW, POS_STOP_F, VEL_STOP_F)

lab = connect("g1", live=LIVE)
gate = MotionGate(lab)

## 1. 先跑通：一条 `LowCmd` 长什么样

每个电机收到 5 个数，电机驱动器内部按下面这条式子算出力矩（每秒算上千次）：

$$\tau = k_p\,(q_{des} - q) + k_d\,(\dot q_{des} - \dot q) + \tau_{ff}$$

| 字段 | 含义 | 只想做位置控制时 |
|---|---|---|
| `q` | 目标位置 [rad] | 目标角 |
| `dq` | 目标速度 [rad/s] | 0 |
| `kp` | 位置刚度 | 几十（G1 官方示例 40~100） |
| `kd` | 速度阻尼 | 1~2 |
| `tau` | 前馈力矩 [N·m] | 0（或重力补偿） |

所以"位置控制 / 速度控制 / 力矩控制"不是三个接口，而是**同一条式子里哪几项非零**：
纯力矩控制就是 `kp = kd = 0`，只给 `tau`。

下面按官方示例（`example/g1/low_level/g1_low_level_example.py`）填一条指令。

In [ ]:
from unitree_sdk2py.core.channel import ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_, LowState_
from unitree_sdk2py.utils.crc import CRC

# 先读一条当前状态: 起点必须是"关节现在在哪"
states = []
sub = ChannelSubscriber("rt/lowstate", LowState_); sub.Init(lambda m: states.append(m), 10)
time.sleep(1.0)
state = states[-1]
q_now = np.array([state.motor_state[i].q for i in range(G1_NUM_MOTOR)])
print("mode_machine (从 LowState 读到):", state.mode_machine)
print("当前关节角 [前 6 个]:", np.round(q_now[:6], 2))

crc = CRC()
cmd = unitree_hg_msg_dds__LowCmd_()
cmd.mode_pr = 0                                  # 0 = PR 模式: 踝关节按 pitch/roll 控制 (1 = AB: 按两根推杆)
cmd.mode_machine = state.mode_machine            # 必须原样抄回 LowState 里的值
for i in range(G1_NUM_MOTOR):
    m = cmd.motor_cmd[i]
    m.mode = 1                                   # 1 = 使能, 0 = 失能
    m.q, m.dq, m.tau = float(q_now[i]), 0.0, 0.0  # 目标 = 当前位置: "保持不动"
    m.kp, m.kd = float(G1_KP[i]), float(G1_KD[i])
cmd.crc = crc.Crc(cmd)
print(f"LowCmd: 电机槽位 {len(cmd.motor_cmd)} 个 (用前 {G1_NUM_MOTOR} 个), crc = {cmd.crc:#010x}")

三个容易漏的地方：

1. **`mode_machine`**：G1 有多种构型（23/29 自由度、腰锁不锁……），这个数标识机型。官方示例的做法是**先收到一条 `LowState`，把里面的 `mode_machine` 抄进 `LowCmd`**，收到之前一条指令都不发。
2. **CRC**：指令末尾的校验码。机器人用它判断这条指令在传输中有没有坏。
3. **起点 = 当前位置**：第一条指令的目标必须是关节**此刻**所在的位置。

### CRC 是怎么算的

02 节说过会再遇到 `struct`。SDK 用一个格式串把整条 `LowCmd` 打包成字节，再对这些字节算 CRC32
（出处：`unitree_sdk2py/utils/crc.py`）：

In [ ]:
FMT_HG_LOWCMD = "<2B2x" + "B3x5fI" * 35 + "5I"       # mode_pr, mode_machine, 35 x (mode,q,dq,tau,kp,kd,reserve), reserve[4], crc
FMT_GO_LOWCMD = "<4B4IH2x" + "B3x5f3I" * 20 + "4B" + "55Bx2I"
print("G1 LowCmd 打包后:", struct.calcsize(FMT_HG_LOWCMD), "字节;  Go2 LowCmd:", struct.calcsize(FMT_GO_LOWCMD), "字节")

before = cmd.crc
cmd.motor_cmd[3].q += 0.001                       # 只把左膝目标改 0.001 rad
print(f"改动前 crc = {before:#010x}\n改动后 crc = {crc.Crc(cmd):#010x}   <- 内容变了，校验码必须重算")
print(f"而指令里还留着旧的 crc = {cmd.crc:#010x} —— 这样一条指令发出去，校验对不上，会被机器人丢弃")
cmd.motor_cmd[3].q -= 0.001; cmd.crc = crc.Crc(cmd)

**停一下：** 忘了重算 CRC 的症状是什么？——**机器人毫无反应，你的程序也不报错**。和 05 节"类型不对"一样属于静默失败。
所以循环里永远是：改完所有字段 → 最后一行算 CRC → 发布。

## 2. 拆开：`kp`、`kd`、`tau` 各管什么

没有机器人可以试，我们用一组**对照实验的记录**（`data/lowlevel_runs.npz`，**合成**：简化的"单关节惯量 + 摩擦 + 负载"模型，500 Hz）。
每组实验给同一个关节同一条目标轨迹，只换一个条件。

### 实验 A：G1 抬左臂——目标平滑还是跳变 × 有没有 `kd`

In [ ]:
runs = load_npz("lowlevel_runs")
J = G1_JOINT_ID["left_shoulder_pitch"]
CASES = (("smooth_kd", "smooth target, kd=1"), ("smooth_kd0", "smooth target, kd=0"),
         ("step_kd", "target jumps, kd=1"), ("step_kd0", "target jumps, kd=0"))
fig, ax = plt.subplots(2, 4, figsize=(13, 5), sharex=True)
for k, (name, title) in enumerate(CASES):
    t = runs[f"g1_{name}_t"]
    ax[0, k].plot(t, runs[f"g1_{name}_q_des"][:, J], "r--", label="target"); ax[0, k].plot(t, runs[f"g1_{name}_q"][:, J], label="actual")
    ax[0, k].set_title(title, fontsize=9); ax[0, k].grid(alpha=.3)
    ax[1, k].plot(t, runs[f"g1_{name}_tau"][:, J], "g", lw=.8); ax[1, k].grid(alpha=.3); ax[1, k].set_xlabel("s")
ax[0, 0].legend(fontsize=8); ax[0, 0].set_ylabel("position [rad]"); ax[1, 0].set_ylabel("torque [N·m]")
plt.tight_layout(); plt.show()

for name, title in CASES:
    q, qd, tau, t = (runs[f"g1_{name}_{k}"] for k in ("q", "q_des", "tau", "t"))
    q, qd, tau = q[:, J], qd[:, J], tau[:, J]
    late = t > 2.8                                   # 目标早已到位之后
    print(f"{title:<22} 超调 {max(0, qd[-1] - q.min()):.3f} rad   到位后仍在晃的幅度 {np.ptp(q[late]):.3f} rad   力矩峰值 {np.abs(tau).max():5.1f} N·m")

**对着图和数字回答：**
1. 第 1、2 列（目标平滑）：位置曲线上 `kd = 0` 和 `kd = 1` **几乎看不出区别**（只有力矩那一行已经在微微发颤）。那 `kd = 0` 是不是也挺好？
2. 第 4 列（目标跳变 + `kd = 0`）：到位后晃了多久？`kd` 这一项在式子里起什么作用？（阻尼：速度越大，反向力矩越大，把振荡"吃掉"。）
3. 第 3 列：有 `kd` 就不怎么晃了，但力矩峰值是第 1 列的多少倍？这一下打在真机的减速器上意味着什么？

第 1 问是这组实验最重要的一课：**平滑的目标会把坏增益藏起来。** 一切正常，直到有人碰了一下机器人、或者目标因为某个 bug 跳了一下——
然后它就像第 4 列那样抖个不停。所以调增益时要故意给一个小的阶跃来"敲一敲"，而正式运行时目标永远要平滑。

### 实验 B：Go2 站起来——到不了位的那 0.1 rad

In [ ]:
C = 2                                               # FR_calf (右前小腿)
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
for name, label in (("no_ff", "tau_ff = 0"), ("good", "tau_ff = load")):
    t = runs[f"go2_{name}_t"]
    ax[0].plot(t, runs[f"go2_{name}_q"][:, C], label=label)
    ax[1].plot(t, runs[f"go2_{name}_tau"][:, C], label=label)
ax[0].plot(t, runs["go2_good_q_des"][:, C], "r--", label="target")
ax[0].set(title="FR_calf position [rad]", xlabel="s"); ax[1].set(title="FR_calf torque [N·m]", xlabel="s")
[a.legend(fontsize=8) or a.grid(alpha=.3) for a in ax]; plt.tight_layout(); plt.show()

err = runs["go2_no_ff_q"][-1, C] - runs["go2_no_ff_q_des"][-1, C]
tau_hold = runs["go2_no_ff_tau"][-1, C]
print(f"无前馈: 稳态误差 {err:+.3f} rad, 此时电机出力 {tau_hold:.1f} N·m, kp = 60  ->  力矩 / kp = {tau_hold / 60:.3f} rad")
print(f"有前馈: 稳态误差 {runs['go2_good_q'][-1, C] - runs['go2_good_q_des'][-1, C]:+.3f} rad")

这就是开头那个问题的答案。站起来后小腿电机要一直出力撑住身体。在 PD 控制里，**力矩只能来自误差**：
`τ = kp × 误差`，所以要出 6 N·m 的力、`kp` 是 60，就**必须**留 0.1 rad 的误差。

两种消除办法：
- 加大 `kp`（误差变小，但更硬、更容易振荡、碰到东西更危险）；
- **把已知的负载力矩填进 `tau`（前馈）**，让 PD 只负责修正剩下的小误差。这是正路。

> 这里的负载模型是合成的简化（小腿恒定 6 N·m）。真机的重力矩随姿态变化，需要用动力学模型计算——这超出本课范围，但"前馈 + 小增益反馈"的思路不变。

## 3. 三道保险：限位、插值、限速

实验 A 的右列告诉我们目标不能跳。公共库的 `SafeJointCommander`（`common/joints.py`）把三件事串起来：
① 目标超出关节限位 → 夹回去；② 起点到终点 → 平滑插值；③ 给的时间太短 → 自动拉长到不超速。

**预测**：下面故意给一个越界的目标（左肘 3.0 rad，上限约 2.09）和一个太短的时间（0.2 s）。它会怎么处理？

In [ ]:
from common.joints import SafeJointCommander

lower, upper = g1_limits()
commander = SafeJointCommander(lower, upper, dt=0.002, max_vel=2.0)
goal = q_now.copy()
goal[G1_JOINT_ID["left_elbow"]] = 3.0               # 越界
goal[G1_JOINT_ID["left_shoulder_pitch"]] = -1.2
T = commander.start(q_now, goal, duration=0.2)      # 太短
print("被夹住的关节:", [G1_JOINTS[i] for i in commander.clipped_joints],
      f"-> {commander.q_goal[G1_JOINT_ID['left_elbow']]:.2f} rad (上限 {upper[G1_JOINT_ID['left_elbow']]:.2f} 再留 0.05 余量)")
print(f"要求 0.2 s, 实际安排 {T:.2f} s")
traj = []
while not commander.done:
    traj.append(commander.step())
traj = np.array(traj)
vel = np.abs(np.diff(traj, axis=0)).max() / 0.002
print(f"轨迹 {len(traj)} 拍, 最大关节速度 {vel:.2f} rad/s (上限 2.0)")

## 4. 完整的控制循环（默认不下发）

把前面的零件装起来，就是官方示例的结构：**释放运控 → 等第一条状态 → 定时循环（算目标 → 填指令 → CRC → 发布）**。
官方用 `RecurrentThread(interval=0.002)` 做 500 Hz 定时，这里照用。

In [ ]:
from unitree_sdk2py.comm.motion_switcher.motion_switcher_client import MotionSwitcherClient
from unitree_sdk2py.utils.thread import RecurrentThread


class LowLevelArmDemo:
    """吊架上的 G1: 其余关节保持不动，把左臂平滑抬到目标位置。"""

    def __init__(self, goal_offsets: dict, duration=3.0, dt=0.002):
        self.dt, self.duration, self.goal_offsets = dt, duration, goal_offsets
        self.state = None
        self.cmd = unitree_hg_msg_dds__LowCmd_()
        self.commander = SafeJointCommander(*g1_limits(), dt=dt, max_vel=2.0)
        self.pub = ChannelPublisher("rt/lowcmd", LowCmd_); self.pub.Init()
        self.sub = ChannelSubscriber("rt/lowstate", LowState_); self.sub.Init(self.on_state, 10)
        self.log = []

    def on_state(self, msg):
        self.state = msg

    def release_high_level(self):
        """做低层控制前必须先释放运控服务 (官方示例的做法: 反复 ReleaseMode 直到 CheckMode 的 name 为空)。"""
        msc = MotionSwitcherClient(); msc.SetTimeout(5.0); msc.Init()
        gate.call(msc.ReleaseMode)
        if gate.will_send:
            code, result = msc.CheckMode()
            while result and result.get("name"):
                msc.ReleaseMode(); time.sleep(1.0); code, result = msc.CheckMode()

    def start(self):
        while self.state is None:                    # 没收到状态之前，一条指令都不发
            time.sleep(0.05)
        q0 = np.array([self.state.motor_state[i].q for i in range(G1_NUM_MOTOR)])
        goal = q0.copy()
        for name, off in self.goal_offsets.items():
            goal[G1_JOINT_ID[name]] += off
        self.commander.start(q0, goal, self.duration)
        self.thread = RecurrentThread(interval=self.dt, target=self.tick, name="lowlevel")
        self.thread.Start()

    def tick(self):                                   # 每 2 ms 一次
        if self.commander.done:
            q_des = self.commander.q_goal
        else:
            q_des = self.commander.step()
        self.cmd.mode_pr = 0
        self.cmd.mode_machine = self.state.mode_machine
        for i in range(G1_NUM_MOTOR):
            m = self.cmd.motor_cmd[i]
            m.mode, m.q, m.dq, m.tau, m.kp, m.kd = 1, float(q_des[i]), 0.0, 0.0, float(G1_KP[i]), float(G1_KD[i])
        self.cmd.crc = crc.Crc(self.cmd)
        if gate.allow_write("rt/lowcmd"):
            self.pub.Write(self.cmd)
        self.log.append((time.monotonic(), q_des[G1_JOINT_ID["left_shoulder_pitch"]]))

    def stop(self):
        self.thread.Wait(1.0); self.sub.Close(); self.pub.Close()


demo = LowLevelArmDemo({"left_shoulder_pitch": -1.0, "left_elbow": -0.3})
demo.release_high_level()
demo.start()
time.sleep(4.0)
demo.stop()
tl = np.array([x[0] for x in demo.log]); ql = np.array([x[1] for x in demo.log])
dt_ms = np.diff(tl) * 1e3
print(f"循环跑了 {len(tl)} 拍; 周期 平均 {dt_ms.mean():.2f} ms, 最大 {dt_ms.max():.1f} ms (目标 2.00 ms)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 2.6))
ax[0].plot(tl - tl[0], ql); ax[0].set(title="planned left_shoulder_pitch target [rad]", xlabel="s"); ax[0].grid(alpha=.3)
ax[1].hist(dt_ms, bins=60); ax[1].set(title="control loop period [ms]", yscale="log"); ax[1].grid(alpha=.3); plt.show()

**看打印的平均周期和右图**：我们要的是 2 ms（500 Hz）。达到了吗？

多半没有。每一拍要用 Python 填 29 个电机、打包 1004 字节算 CRC，在普通电脑（尤其是虚拟机）上本身就要几毫秒；
偶尔还会有几十毫秒的"卡顿"。上一节录 `arm_sdk` 时见到的"50 Hz 变 41 Hz"是同一件事。这带来两个后果：

- **轨迹变慢了**：`SafeJointCommander` 每调一次 `step()` 前进 2 ms 的"计划时间"。循环实际 4 ms 一拍，3 秒的动作就要 6 秒才做完。
  想让动作按真实时间走，应该用 `time.monotonic()` 算 `t`，而不是数拍子（课后练习 4）。
- **卡顿的那几十毫秒里机器人收不到新指令**，电机按上一条指令继续出力。位置目标平滑时这只是"晚一点到"；如果目标会跳，就是事故。

所以：对周期要求严格的控制（步态、力控）用 C++ 写、跑在机器人本体的电脑上；Python 适合 50~100 Hz 的上层控制和原型验证。

## 5. Go2 的 `LowCmd` 有什么不同

思路完全一样，消息不同（出处：`example/go2/low_level/go2_stand_example.py`）。看官方站立示例的初始化：

In [ ]:
from unitree_sdk2py.idl.default import unitree_go_msg_dds__LowCmd_

go = unitree_go_msg_dds__LowCmd_()
go.head[0], go.head[1] = GO2_LOWCMD_HEAD            # 0xFE, 0xEF: 帧头
go.level_flag = GO2_LEVEL_LOW                       # 0xFF: 声明这是低层指令
for i in range(20):
    go.motor_cmd[i].mode = 0x01                     # FOC 模式
    go.motor_cmd[i].q, go.motor_cmd[i].dq = POS_STOP_F, VEL_STOP_F     # 哨兵值: "这一项不控制"
    go.motor_cmd[i].kp = go.motor_cmd[i].kd = go.motor_cmd[i].tau = 0.0
for i in range(12):                                 # 只有前 12 个是腿上的电机
    go.motor_cmd[i].q, go.motor_cmd[i].dq = float(GO2_POSE_STAND[i]), 0.0
    go.motor_cmd[i].kp, go.motor_cmd[i].kd = 60.0, 5.0
go.crc = crc.Crc(go)
print(f"Go2 LowCmd: head={[hex(h) for h in go.head]} level_flag={go.level_flag:#x} 电机槽位 {len(go.motor_cmd)} crc={go.crc:#010x}")
print("POS_STOP_F =", POS_STOP_F, "  VEL_STOP_F =", VEL_STOP_F)

| | G1 (`unitree_hg/LowCmd`) | Go2 (`unitree_go/LowCmd`) |
|---|---|---|
| 电机槽位 / 实际使用 | 35 / 29 | 20 / 12 |
| 帧头、级别 | 无 | `head = [0xFE, 0xEF]`，`level_flag = 0xFF` |
| 机型标识 | `mode_machine`（从 `LowState` 抄），`mode_pr`（踝关节 PR/AB） | 无 |
| "不控制这一项" | `mode = 0` 失能，或 `kp = 0` | 哨兵值 `PosStopF = 2.146e9`、`VelStopF = 16000` |
| 官方示例增益 | kp 40~100，kd 1~2 | kp 60，kd 5 |
| 打包长度（算 CRC 用） | 1004 字节 | 812 字节 |
| 共同点 | Topic 都是 `rt/lowcmd`；控制律相同；都要 CRC；都要先 `ReleaseMode` | |

In [ ]:
sub.Close(); lab.close()

## 上真机怎么跑
1. G1 挂上保护吊架，脚离地；确认遥控器能进阻尼。
2. `LIVE = True`；先只运行到"读一条当前状态"，确认 `mode_machine` 和关节角合理。
3. `gate.confirm("我已清场并手握急停")`，然后 `demo.release_high_level()`——**此刻机器人会失去支撑力**，这是预期的。
4. `demo.start()`。第一次把 `goal_offsets` 的幅度减到 0.3 rad。
5. 结束后用遥控器或 `MotionSwitcherClient.SelectMode(之前查到的模式名)` 恢复运控。

## 小结
- 一条控制律：`τ = kp·(q_des − q) + kd·(dq_des − dq) + τ_ff`。位置/速度/力矩控制是它的三种用法。
- `kd` 太小 → 一受扰动就振荡（目标平滑时看不出来！）；目标跳变 → 力矩尖峰；有负载无前馈 → 稳态误差 = 负载 / kp。
- 合法指令的三要素：`mode_machine` 抄回、起点 = 当前位置、**最后算 CRC**。
- 三道保险：限位、插值、限速（`SafeJointCommander`）。

## 课后练习
1. 实验 B 里把 `kp` 从 60 提到 120、不加前馈，稳态误差应该是多少？（先算，再想：为什么不这么做？）
2. 写"纯阻尼"指令：所有关节 `kp = 0, kd = 3, tau = 0`。它对应 09 节的哪个模式？这是低层控制程序退出前应该发的最后一种指令，为什么？
3. 把 `LowLevelArmDemo.tick` 里的 `self.cmd.crc = ...` 一行删掉重跑（离线）。程序报错了吗？这说明这类错误要靠什么来发现？
4. 改写 `LowLevelArmDemo`：不用"数拍子"，而是每拍用 `time.monotonic()` 算出已经过去的真实时间 `t`，再调 `common.joints.interpolate(q0, goal, t, duration)`。重跑后动作用时是多少？

**下一节**：Part 2 收尾——把 G1 也接进 ROS 2，并把一路上零零散散发现的 Go2 / G1 差异汇总成一张表。